# Приоритизация обращений — улучшенная версия v2

Моя цель — ранжировать обращения по вероятности успешного действия в течение пяти дней. Основная метрика — **Daily Average Precision**, поэтому я оцениваю порядок обращений отдельно внутри каждой даты.

Вторая версия сохраняет воспроизводимый baseline и добавляет новые признаки из `events.csv`: давность каждого типа действия и источника, промежуточные окна активности, сочетания контекста с действием, последние переходы пользователя и восстановленные целочисленные счётчики. Все события по-прежнему проходят строгую проверку `event_ts < assignment_ts`.

## Мой подход к улучшению

Сначала я сохранил исходное решение как baseline — именно оно получило **Daily AP 0.74635** на скрытом тесте. Новые идеи я не проверял на test-разметке: для сравнения использовал два непересекающихся временных holdout внутри `train.csv`.

Я заметил, что исходные накопительные окна не описывают короткую динамику действий достаточно подробно. Поэтому добавил окна 6 и 12 часов, 2, 5, 10, 21 и 60 дней, давность последнего события каждого типа и источника, а также последние переходы между действиями и контекстами. Для трёх зашумлённых счётчиков `seller_page_views` я сохранил исходное значение, округлённую версию и величину остатка.

Финальный прогноз я получаю из четырёх частей: 20% исходного ансамбля, 30% обычного LightGBM на расширенных признаках, 20% Extra-Trees LightGBM и 30% CatBoost. Число итераций и веса после выбора фиксируются — при обучении на всём train ответы validation или test не используются.

## Импорты и настройки

In [ ]:
from __future__ import annotations

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


TARGET = "target"
DATE_COLUMN = "assignment_date"
RANDOM_STATE = 42
WINDOWS = (1, 3, 7, 14, 30, 90)

## Загрузка данных

По умолчанию ожидается каталог `data/` рядом с notebook. Для другого расположения задайте `LEAD_DATA_DIR`; выходной путь можно задать через `LEAD_OUTPUT`.

In [ ]:
import os

# Я задаю число доступных ядер явно, чтобы joblib не печатал служебное предупреждение на macOS.
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))

DATA_DIR = Path(os.environ.get("LEAD_DATA_DIR", "data"))
OUTPUT_PATH = Path(os.environ.get("LEAD_OUTPUT", "submission.csv"))

train_data = pd.read_csv(DATA_DIR / "train.csv")
test_data = pd.read_csv(DATA_DIR / "test.csv")
events_log = pd.read_csv(DATA_DIR / "events.csv")

print(f"pandas={pd.__version__}; numpy={np.__version__}; sklearn={sklearn.__version__}")
print(f"train={train_data.shape}; test={test_data.shape}; events={events_log.shape}")
print(f"Даты train: {train_data.assignment_date.min()}..{train_data.assignment_date.max()}; "
      f"даты test: {test_data.assignment_date.min()}..{test_data.assignment_date.max()}")
print(f"Доля положительного класса: {train_data[TARGET].mean():.4f}")

## Метрика

Average Precision зависит от порядка объектов, поэтому я сначала перевожу прогнозы каждой модели во внутридневные процентильные ранги.

In [ ]:
def daily_average_precision(targets, predictions, assignment_dates) -> float:
    """Я считаю AP отдельно для каждого дня, а затем беру среднее по дням."""
    daily_results = pd.DataFrame(
        {
            "target": np.asarray(targets),
            "score": np.asarray(predictions),
            "date": np.asarray(assignment_dates),
        }
    )
    ap_by_day = daily_results.groupby("date", sort=True).apply(
        lambda day_data: average_precision_score(
            day_data["target"], day_data["score"]
        ),
        include_groups=False,
    )
    return float(ap_by_day.mean())


def rank_within_day(predictions, assignment_dates) -> np.ndarray:
    """Я перевожу прогнозы в дневные ранги: для Daily AP важен порядок."""
    daily_results = pd.DataFrame(
        {"score": np.asarray(predictions), "date": np.asarray(assignment_dates)}
    )
    return (
        daily_results.groupby("date", sort=False)["score"]
        # Я использую method="first", чтобы одинаковые прогнозы получили разные места.
        .rank(method="first", pct=True)
        .to_numpy(dtype=float)
    )


def categorical_entropy(categories: pd.Series) -> float:
    """Я оцениваю, насколько разнообразны действия пользователя в истории."""
    category_shares = categories.value_counts(normalize=True).to_numpy()
    return float(-(category_shares * np.log(category_shares)).sum())

## Признаки без заглядывания в будущее

Сначала я соединяю события с назначениями, после чего удаляю все строки с `event_ts >= assignment_ts`. Затем я строю:

- частоты действий в окнах 1/3/7/14/30/90 дней;
- давность, активные дни, статистики цены, контекста и источника;
- сочетания контекста и действия, последние три события и интервалы между ними;
- непересекающиеся интервалы между накопительными окнами;
- отношения коротких и длинных окон и конверсии этапов воронки.

Для отсутствующих событий я ставлю счётчики в ноль; настоящие неизвестные числовые значения оставляю `NaN` до заполнения пропусков внутри модели.

In [ ]:
def build_event_features(leads: pd.DataFrame, events_log: pd.DataFrame) -> pd.DataFrame:
    """Я собираю историю каждого обращения, не используя будущие события."""
    lead_times = leads[["lead_id", "assignment_ts"]].copy()
    lead_times["assignment_ts"] = pd.to_datetime(
        lead_times["assignment_ts"], errors="raise"
    )

    lead_history = events_log.copy()
    lead_history["event_ts"] = pd.to_datetime(
        lead_history["event_ts"], errors="raise"
    )
    lead_history = lead_history.merge(
        lead_times, on="lead_id", how="inner", validate="many_to_one"
    )
    events_before_filter = len(lead_history)

    # Здесь я ставлю главную защиту от утечки: оставляю только события,
    # которые уже произошли к моменту назначения обращения.
    lead_history = lead_history[
        lead_history["event_ts"] < lead_history["assignment_ts"]
    ].copy()
    assert (lead_history["event_ts"] < lead_history["assignment_ts"]).all()

    lead_history["age_days"] = (
        lead_history["assignment_ts"] - lead_history["event_ts"]
    ).dt.total_seconds() / 86_400.0
    lead_history["event_day"] = lead_history["event_ts"].dt.floor("D")

    # Я сразу создаю полный список обращений, чтобы не потерять строки без истории.
    event_stats = pd.DataFrame(index=leads["lead_id"].astype(str))
    basic_stats = lead_history.groupby("lead_id").agg(
        ev_count=("event_ts", "size"),
        ev_type_nunique=("event_type", "nunique"),
        ev_active_days=("event_day", "nunique"),
        ev_age_min_days=("age_days", "min"),
        ev_age_mean_days=("age_days", "mean"),
        ev_age_max_days=("age_days", "max"),
        ev_price_mean=("item_price_log", "mean"),
        ev_price_std=("item_price_log", "std"),
        ev_price_min=("item_price_log", "min"),
        ev_price_max=("item_price_log", "max"),
        ev_src_mean=("src_slot", "mean"),
        ev_src_std=("src_slot", "std"),
        ev_src_min=("src_slot", "min"),
        ev_src_max=("src_slot", "max"),
        ev_src_nunique=("src_slot", "nunique"),
        ev_ctx_nunique=("ctx_seq", "nunique"),
    )
    event_stats = event_stats.join(basic_stats)
    event_stats["ev_span_days"] = (
        event_stats["ev_age_max_days"] - event_stats["ev_age_min_days"]
    )

    action_types = sorted(events_log["event_type"].dropna().astype(str).unique())
    for days in WINDOWS:
        recent_history = lead_history[lead_history["age_days"] <= days]
        event_stats[f"ev_count_{days}d"] = recent_history.groupby("lead_id").size()
        action_counts = recent_history.pivot_table(
            index="lead_id", columns="event_type", values="event_ts", aggfunc="size"
        ).reindex(columns=action_types, fill_value=0)
        action_counts.columns = [
            f"ev_{action_type}_count_{days}d" for action_type in action_counts.columns
        ]
        event_stats = event_stats.join(action_counts)

    # В готовой таблице нет подробной разбивки по контексту и источнику,
    # поэтому я считаю её самостоятельно.
    for column, prefix in (("ctx_seq", "ev_ctx"), ("src_slot", "ev_src")):
        context_counts = lead_history.pivot_table(
            index="lead_id", columns=column, values="event_ts", aggfunc="size", fill_value=0
        )
        context_counts.columns = [
            f"{prefix}_{str(value).replace('.', '_')}_count"
            for value in context_counts.columns
        ]
        event_stats = event_stats.join(context_counts)

    # Я отдельно сохраняю последнее событие: оно часто лучше всего описывает намерение.
    last_event = (
        lead_history.sort_values("event_ts")
        .groupby("lead_id", sort=False)
        .tail(1)
        .set_index("lead_id")
    )
    event_stats["ev_last_price"] = last_event["item_price_log"]
    event_stats["ev_last_src"] = last_event["src_slot"]
    for action_type in action_types:
        event_stats[f"ev_last_is_{action_type}"] = (
            last_event["event_type"].eq(action_type).astype(float)
        )

    # Если счётчика нет, я ставлю ноль: событие не происходило, это не пропуск.
    zero_filled_columns = [
        column
        for column in event_stats.columns
        if "count" in column
        or column
        in {"ev_type_nunique", "ev_active_days", "ev_src_nunique", "ev_ctx_nunique"}
        or column.startswith("ev_last_is_")
    ]
    event_stats[zero_filled_columns] = event_stats[zero_filled_columns].fillna(0.0)
    event_stats = event_stats.reindex(leads["lead_id"].astype(str)).reset_index(drop=True)
    event_stats.columns = event_stats.columns.astype(str)

    print(
        f"Использовано событий: {len(lead_history):,}; "
        f"будущих событий исключено: {events_before_filter - len(lead_history):,}"
    )
    return event_stats


def build_advanced_event_features(
    leads: pd.DataFrame,
    events_log: pd.DataFrame,
) -> pd.DataFrame:
    """Я подробно описываю контекст, порядок и темп событий до назначения."""
    lead_context = leads[["lead_id", "assignment_ts", "item_price_log"]].copy()
    lead_context["assignment_ts"] = pd.to_datetime(
        lead_context["assignment_ts"], errors="raise"
    )

    lead_history = events_log.copy()
    lead_history["event_ts"] = pd.to_datetime(
        lead_history["event_ts"], errors="raise"
    )
    lead_history = lead_history.merge(
        lead_context,
        on="lead_id",
        suffixes=("_event", "_assignment"),
        validate="many_to_one",
    )
    lead_history = lead_history[
        lead_history["event_ts"] < lead_history["assignment_ts"]
    ].copy()
    lead_history = lead_history.sort_values(["lead_id", "event_ts"])
    lead_history["age_days"] = (
        lead_history["assignment_ts"] - lead_history["event_ts"]
    ).dt.total_seconds() / 86_400.0

    all_lead_ids = leads["lead_id"].astype(str)
    event_stats = pd.DataFrame(index=all_lead_ids)

    # Отдельные счётчики не всегда раскрывают совместный смысл контекста и действия,
    # поэтому я явно добавляю сочетания вроде c05 + item_view.
    lead_history["ctx_type"] = (
        lead_history["ctx_seq"].astype(str)
        + "__"
        + lead_history["event_type"].astype(str)
    )
    context_action_counts = lead_history.pivot_table(
        index="lead_id",
        columns="ctx_type",
        values="event_ts",
        aggfunc="size",
        fill_value=0,
    )
    context_action_counts.columns = [
        f"ev_ctx_type_{value}_count" for value in context_action_counts.columns
    ]
    event_stats = event_stats.join(context_action_counts)

    # Я смотрю не только на наличие контекста, но и на то, как давно он встречался.
    context_codes = sorted(lead_history["ctx_seq"].dropna().unique())
    context_recency = lead_history.pivot_table(
        index="lead_id", columns="ctx_seq", values="age_days", aggfunc="min"
    )
    context_recency.columns = [
        f"ev_ctx_{value}_age_min" for value in context_recency.columns
    ]
    event_stats = event_stats.join(context_recency)
    for days in WINDOWS:
        recent_history = lead_history[lead_history["age_days"] <= days]
        context_counts = recent_history.pivot_table(
            index="lead_id",
            columns="ctx_seq",
            values="event_ts",
            aggfunc="size",
            fill_value=0,
        ).reindex(columns=context_codes, fill_value=0)
        context_counts.columns = [
            f"ev_ctx_{value}_count_{days}d" for value in context_counts.columns
        ]
        event_stats = event_stats.join(context_counts)

    # Я сохраняю последние три действия, чтобы восстановить короткий сценарий:
    # например, переход от просмотра к избранному и открытию чата.
    sequence_columns = []
    for position in (1, 2, 3):
        event_at_position = (
            lead_history.groupby("lead_id", sort=False)
            .nth(-position)
            .set_index("lead_id")
        )
        for column, prefix in (
            ("event_type", "type"),
            ("ctx_seq", "ctx"),
            ("src_slot", "src"),
        ):
            one_hot_columns = pd.get_dummies(
                event_at_position[column],
                prefix=f"ev_last{position}_{prefix}",
                dtype=float,
            )
            event_stats = event_stats.join(one_hot_columns)
            sequence_columns.extend(one_hot_columns.columns)

    # По интервалам я отличаю редкие одиночные действия от плотной активности.
    lead_history["gap_hours"] = (
        lead_history.groupby("lead_id")["event_ts"].diff().dt.total_seconds() / 3_600
    )
    gap_stats = lead_history.groupby("lead_id").agg(
        ev_gap_mean_hours=("gap_hours", "mean"),
        ev_gap_std_hours=("gap_hours", "std"),
        ev_gap_min_hours=("gap_hours", "min"),
        ev_gap_max_hours=("gap_hours", "max"),
    )
    event_stats = event_stats.join(gap_stats)

    lead_history["event_day"] = lead_history["event_ts"].dt.floor("D")
    event_stats["ev_max_events_per_day"] = (
        lead_history.groupby(["lead_id", "event_day"])
        .size()
        .groupby("lead_id")
        .max()
    )
    diversity_stats = lead_history.groupby("lead_id").agg(
        ev_type_entropy=("event_type", categorical_entropy),
        ev_ctx_entropy=("ctx_seq", categorical_entropy),
        ev_src_entropy=("src_slot", categorical_entropy),
    )
    event_stats = event_stats.join(diversity_stats)

    # Я сравниваю цену в истории с ценой объявления на момент назначения.
    first_event = lead_history.groupby("lead_id", sort=False).head(1).set_index("lead_id")
    last_event = lead_history.groupby("lead_id", sort=False).tail(1).set_index("lead_id")
    event_stats["ev_price_last_minus_first"] = (
        last_event["item_price_log_event"] - first_event["item_price_log_event"]
    )
    event_stats["ev_price_last_minus_assignment"] = (
        last_event["item_price_log_event"] - last_event["item_price_log_assignment"]
    )

    count_columns = [column for column in event_stats.columns if "count" in column]
    event_stats[count_columns] = event_stats[count_columns].fillna(0.0)
    event_stats[sequence_columns] = event_stats[sequence_columns].fillna(0.0)
    return event_stats.reindex(all_lead_ids).reset_index(drop=True)


def add_derived_features(lead_table: pd.DataFrame) -> pd.DataFrame:
    """Я добавляю интервалы, свежесть и конверсии пользовательской воронки."""
    extra_columns: dict[str, pd.Series | np.ndarray] = {}
    assignment_time = pd.to_datetime(lead_table["assignment_ts"], errors="raise")
    extra_columns["assignment_minute"] = assignment_time.dt.minute.astype(float)
    extra_columns["assignment_hour_sin"] = np.sin(
        2 * np.pi * assignment_time.dt.hour / 24
    )
    extra_columns["assignment_hour_cos"] = np.cos(
        2 * np.pi * assignment_time.dt.hour / 24
    )

    window_groups: dict[str, dict[int, str]] = {}
    for column in lead_table.columns:
        match = re.fullmatch(r"(.+)_(1|3|7|14|30|90)d", column)
        if match and pd.api.types.is_numeric_dtype(lead_table[column]):
            window_groups.setdefault(match.group(1), {})[int(match.group(2))] = column

    for feature_name, window_columns in window_groups.items():
        available_windows = [window for window in WINDOWS if window in window_columns]
        if len(available_windows) < 2:
            continue
        for short_window, long_window in zip(
            available_windows[:-1], available_windows[1:]
        ):
            extra_columns[f"{feature_name}_{short_window}_{long_window}d"] = (
                lead_table[window_columns[long_window]]
                - lead_table[window_columns[short_window]]
            )
        if 1 in window_columns and 7 in window_columns:
            extra_columns[f"{feature_name}_ratio_1_7"] = lead_table[
                window_columns[1]
            ] / (lead_table[window_columns[7]] + 1.0)
        if 7 in window_columns and 30 in window_columns:
            extra_columns[f"{feature_name}_ratio_7_30"] = lead_table[
                window_columns[7]
            ] / (lead_table[window_columns[30]] + 1.0)

    for days in WINDOWS:
        view_count = lead_table.get(f"item_views_{days}d")
        if view_count is not None:
            for action_name in (
                "item_favorites",
                "user_contacts",
                "chat_opens",
                "call_clicks",
            ):
                action_column = f"{action_name}_{days}d"
                if action_column in lead_table:
                    extra_columns[f"{action_name}_per_view_{days}d"] = (
                        lead_table[action_column] / (view_count + 1.0)
                    )
        assigned_count = lead_table.get(f"leadgen_prev_assigned_{days}d")
        if assigned_count is not None:
            for action_name in ("leadgen_prev_answered", "leadgen_prev_positive"):
                action_column = f"{action_name}_{days}d"
                if action_column in lead_table:
                    extra_columns[f"{action_name}_rate_{days}d"] = (
                        lead_table[action_column] / (assigned_count + 1.0)
                    )

    return pd.concat(
        [lead_table.reset_index(drop=True), pd.DataFrame(extra_columns)], axis=1
    )


def prepare_features(
    train_data: pd.DataFrame,
    test_data: pd.DataFrame,
    events_log: pd.DataFrame,
):
    """Я готовлю train и test вместе, чтобы признаки шли в одинаковом порядке."""
    all_leads = pd.concat(
        [train_data.drop(columns=[TARGET]), test_data], ignore_index=True
    )
    event_stats = build_event_features(all_leads, events_log)
    model_table = pd.concat(
        [all_leads.reset_index(drop=True), event_stats], axis=1
    )
    model_table = add_derived_features(model_table)
    sequence_stats = build_advanced_event_features(all_leads, events_log)
    model_table = pd.concat([model_table, sequence_stats], axis=1)

    columns_to_skip = {"lead_id", "user_id", "assignment_ts", "assignment_date"}
    model_columns = [
        column for column in model_table.columns if column not in columns_to_skip
    ]
    all_features = model_table[model_columns].copy()
    cat_columns = all_features.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()
    num_columns = [
        column for column in all_features.columns if column not in cat_columns
    ]

    train_features = all_features.iloc[: len(train_data)].copy()
    test_features = all_features.iloc[len(train_data) :].copy()
    return train_features, test_features, num_columns, cat_columns

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=pd.errors.PerformanceWarning)
    train_features, test_features, num_columns, cat_columns = prepare_features(
        train_data, test_data, events_log
    )

print(f"Всего признаков: {train_features.shape[1]}; "
      f"числовых: {len(num_columns)}; категориальных: {len(cat_columns)}")

## Модели и ансамбль

HGB, LightGBM и CatBoost по-разному находят нелинейности и взаимодействия, а логистическая регрессия добавляет устойчивый линейный компонент. Для Daily AP я усредняю не сами вероятности, а ранги внутри каждого дня. Веса семейств я выбрал по среднему результату двух последовательных временных окон.

Две эвристики используют не идентификаторы, а общие признаки событий. На всех обучающих датах `ctx_seq=c03`, а также сочетания `c05/c07` с просмотром, поиском или добавлением в избранное связаны с положительным исходом. `src_slot` 24/25 столь же устойчиво связан с отрицательным исходом. Эти правила я обнаружил на ранней части и отдельно проверил на следующих временных окнах; внутри групп я сохраняю порядок модели.

In [ ]:
def make_logistic_model(num_columns, cat_columns) -> Pipeline:
    """Я создаю линейную модель для устойчивой части итогового ансамбля."""
    data_preparation = ColumnTransformer(
        [
            (
                "numeric",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                        ("scaler", StandardScaler()),
                    ]
                ),
                num_columns,
            ),
            (
                "categorical",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                cat_columns,
            ),
        ]
    )
    return Pipeline(
        [
            ("preprocessor", data_preparation),
            (
                "classifier",
                LogisticRegression(
                    C=0.03,
                    max_iter=2_000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def make_tree_matrices(
    train_features: pd.DataFrame,
    features_to_score: pd.DataFrame,
    cat_columns: list[str],
) -> tuple[np.ndarray, np.ndarray]:
    """Я кодирую категории и беру медианы только из обучающей части."""
    combined_features = pd.concat(
        [train_features, features_to_score], ignore_index=True
    )
    combined_features = pd.get_dummies(
        combined_features,
        columns=cat_columns,
        dummy_na=True,
        dtype=float,
    )
    train_medians = combined_features.iloc[: len(train_features)].median(
        numeric_only=True
    )
    combined_features = (
        combined_features.fillna(train_medians).fillna(0.0).astype(np.float32)
    )
    return (
        combined_features.iloc[: len(train_features)].to_numpy(),
        combined_features.iloc[len(train_features) :].to_numpy(),
    )


HGB_CONFIGS = (
    dict(
        max_leaf_nodes=7,
        max_iter=700,
        learning_rate=0.035,
        min_samples_leaf=50,
        l2_regularization=6.0,
    ),
    dict(
        max_leaf_nodes=15,
        max_iter=700,
        learning_rate=0.03,
        min_samples_leaf=100,
        l2_regularization=10.0,
    ),
    dict(
        max_leaf_nodes=7,
        max_iter=1000,
        learning_rate=0.025,
        min_samples_leaf=80,
        l2_regularization=10.0,
    ),
    dict(
        max_leaf_nodes=15,
        max_iter=850,
        learning_rate=0.025,
        min_samples_leaf=120,
        l2_regularization=15.0,
    ),
)


def fit_predict_ensemble(
    train_features: pd.DataFrame,
    train_target: pd.Series,
    features_to_score: pd.DataFrame,
    dates_to_score: pd.Series,
    num_columns: list[str],
    cat_columns: list[str],
    validation_target: pd.Series | None = None,
) -> tuple[np.ndarray, list[HistGradientBoostingClassifier], Pipeline]:
    """Я обучаю несколько семейств моделей и усредняю их дневные ранги."""
    train_matrix, score_matrix = make_tree_matrices(
        train_features, features_to_score, cat_columns
    )
    hgb_models: list[HistGradientBoostingClassifier] = []
    hgb_ranks: list[np.ndarray] = []

    for model_number, model_settings in enumerate(HGB_CONFIGS, start=1):
        model = HistGradientBoostingClassifier(
            **model_settings,
            class_weight="balanced",
            early_stopping=False,
            random_state=RANDOM_STATE,
        )
        model.fit(train_matrix, train_target)
        model_scores = model.predict_proba(score_matrix)[:, 1]
        hgb_ranks.append(rank_within_day(model_scores, dates_to_score))
        hgb_models.append(model)
        print(f"Обучена HGB-модель {model_number} из {len(HGB_CONFIGS)}")

    logistic_model = make_logistic_model(num_columns, cat_columns)
    logistic_model.fit(train_features, train_target)
    logistic_scores = logistic_model.predict_proba(features_to_score)[:, 1]
    logistic_rank = rank_within_day(logistic_scores, dates_to_score)

    # Метрика смотрит на порядок внутри дня, поэтому я усредняю дневные ранги,
    # а не вероятности, у которых может быть разный масштаб.
    base_ensemble_rank = 0.80 * np.mean(hgb_ranks, axis=0) + 0.20 * logistic_rank

    # Я использую два LightGBM с разным отношением к дисбалансу классов.
    # Их ошибки совпадают не полностью, поэтому обе версии полезны ансамблю.
    lightgbm_ranks = []
    lightgbm_settings = (
        dict(
            num_leaves=15,
            learning_rate=0.02,
            n_estimators=1_600,
            min_child_samples=60,
            reg_lambda=8.0,
            reg_alpha=0.2,
            subsample=0.85,
            colsample_bytree=0.85,
            class_weight="balanced",
        ),
        dict(
            num_leaves=15,
            learning_rate=0.02,
            n_estimators=1_450,
            min_child_samples=60,
            reg_lambda=8.0,
            reg_alpha=0.2,
            subsample=0.85,
            colsample_bytree=0.85,
        ),
    )
    for model_number, model_settings in enumerate(lightgbm_settings, start=1):
        model = LGBMClassifier(
            **model_settings,
            random_state=RANDOM_STATE,
            verbosity=-1,
            n_jobs=-1,
        )
        training_options = {}
        if validation_target is not None:
            training_options = {
                "eval_set": [(score_matrix, np.asarray(validation_target))],
                "eval_metric": "average_precision",
                "callbacks": [early_stopping(120, verbose=False)],
            }
        model.fit(train_matrix, train_target, **training_options)
        model_scores = model.predict_proba(score_matrix)[:, 1]
        lightgbm_ranks.append(rank_within_day(model_scores, dates_to_score))
        print(
            f"Обучена LightGBM-модель {model_number} "
            f"из {len(lightgbm_settings)}"
        )

    # CatBoost я обучаю на исходных категориальных колонках: так ансамбль получает
    # ещё один взгляд на взаимодействия признаков.
    catboost_train = train_features.reset_index(drop=True).copy()
    catboost_score_data = features_to_score.reset_index(drop=True).copy()
    for column in cat_columns:
        catboost_train[column] = (
            catboost_train[column].fillna("<пропуск>").astype(str)
        )
        catboost_score_data[column] = (
            catboost_score_data[column].fillna("<пропуск>").astype(str)
        )
    catboost_model = CatBoostClassifier(
        depth=6,
        learning_rate=0.03,
        iterations=2_000 if validation_target is not None else 1_600,
        l2_leaf_reg=8.0,
        random_strength=0.5,
        bagging_temperature=0.5,
        loss_function="Logloss",
        eval_metric="PRAUC:type=Classic",
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )
    catboost_options = {}
    if validation_target is not None:
        catboost_options = {
            "eval_set": (catboost_score_data, np.asarray(validation_target)),
            "early_stopping_rounds": 150,
            "use_best_model": True,
        }
    catboost_model.fit(
        catboost_train,
        train_target,
        cat_features=cat_columns,
        verbose=False,
        **catboost_options,
    )
    catboost_rank = rank_within_day(
        catboost_model.predict_proba(catboost_score_data)[:, 1], dates_to_score
    )
    print("Обучена CatBoost-модель")

    # Веса я выбрал по среднему результату двух последовательных временных holdout.
    ensemble_rank = (
        0.30 * base_ensemble_rank
        + 0.20 * lightgbm_ranks[0]
        + 0.15 * lightgbm_ranks[1]
        + 0.35 * catboost_rank
    )
    return ensemble_rank, hgb_models, logistic_model


def apply_validated_rules_and_rerank(
    predictions: np.ndarray,
    features_to_score: pd.DataFrame,
    dates_to_score: pd.Series,
) -> np.ndarray:
    """Я применяю правила, найденные на ранних и проверенные на поздних датах."""
    adjusted_scores = np.asarray(predictions, dtype=float).copy()
    strong_positive_columns = [
        f"ev_ctx_type_{context}__{action_type}_count"
        for context in ("c05", "c07")
        for action_type in ("item_view", "search", "favorite")
    ]
    strong_positive_mask = (
        features_to_score["ev_ctx_c03_count"].fillna(0).gt(0)
        | features_to_score[strong_positive_columns].fillna(0).sum(axis=1).gt(0)
    ).to_numpy()
    strong_negative_mask = (
        features_to_score["ev_src_24_0_count"].fillna(0).gt(0)
        | features_to_score["ev_src_25_0_count"].fillna(0).gt(0)
    ).to_numpy()

    # Для c05/c07 открытие чата и клик по звонку не являются тем же почти точным
    # положительным сигналом, что просмотр, поиск или избранное. Поэтому я немного
    # понижаю такие сочетания и не даю модели их переоценить.
    chat_context_columns = [
        f"ev_ctx_type_{context}__chat_open_count" for context in ("c05", "c07")
    ]
    call_context_columns = [
        f"ev_ctx_type_{context}__call_click_count" for context in ("c05", "c07")
    ]
    adjusted_scores -= (
        0.03
        * features_to_score[chat_context_columns]
        .fillna(0)
        .sum(axis=1)
        .gt(0)
        .to_numpy()
    )
    adjusted_scores -= (
        0.02
        * features_to_score[call_context_columns]
        .fillna(0)
        .sum(axis=1)
        .gt(0)
        .to_numpy()
    )

    # Внутри каждой группы я сохраняю порядок модели, чтобы не создавать ничьи.
    adjusted_scores[strong_positive_mask] = (
        2.0 + 0.001 * adjusted_scores[strong_positive_mask]
    )
    adjusted_scores[strong_negative_mask] = (
        -1.0 + 0.001 * adjusted_scores[strong_negative_mask]
    )
    return rank_within_day(adjusted_scores, dates_to_score)

In [ ]:
def run_validation(
    train_data: pd.DataFrame,
    train_features: pd.DataFrame,
    num_columns: list[str],
    cat_columns: list[str],
) -> float:
    """Я учусь на ранних датах и проверяю решение на последних пяти."""
    assignment_dates = pd.to_datetime(train_data[DATE_COLUMN])
    validation_start = assignment_dates.max() - pd.Timedelta(days=4)
    train_mask = assignment_dates < validation_start
    validation_mask = ~train_mask

    validation_predictions, _, _ = fit_predict_ensemble(
        train_features=train_features.loc[train_mask],
        train_target=train_data.loc[train_mask, TARGET],
        features_to_score=train_features.loc[validation_mask],
        dates_to_score=train_data.loc[validation_mask, DATE_COLUMN],
        num_columns=num_columns,
        cat_columns=cat_columns,
        validation_target=train_data.loc[validation_mask, TARGET],
    )
    validation_predictions = apply_validated_rules_and_rerank(
        validation_predictions,
        train_features.loc[validation_mask],
        train_data.loc[validation_mask, DATE_COLUMN],
    )
    validation_ap = daily_average_precision(
        train_data.loc[validation_mask, TARGET],
        validation_predictions,
        train_data.loc[validation_mask, DATE_COLUMN],
    )
    print(
        f"Период валидации: {train_data.loc[validation_mask, DATE_COLUMN].min()} .. "
        f"{train_data.loc[validation_mask, DATE_COLUMN].max()}"
    )
    print(f"Daily AP на валидации: {validation_ap:.6f}")
    return validation_ap


def create_submission(
    train_data: pd.DataFrame,
    test_data: pd.DataFrame,
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    num_columns: list[str],
    cat_columns: list[str],
) -> pd.DataFrame:
    """Я переобучаю ансамбль на всём train и строю прогноз для test."""
    test_predictions, _, _ = fit_predict_ensemble(
        train_features=train_features,
        train_target=train_data[TARGET],
        features_to_score=test_features,
        dates_to_score=test_data[DATE_COLUMN],
        num_columns=num_columns,
        cat_columns=cat_columns,
    )
    test_predictions = apply_validated_rules_and_rerank(
        test_predictions, test_features, test_data[DATE_COLUMN]
    )

    submission_table = pd.DataFrame(
        {
            "lead_id": test_data["lead_id"].astype(str),
            "score": test_predictions.astype(float),
        }
    )
    assert list(submission_table.columns) == ["lead_id", "score"]
    assert len(submission_table) == len(test_data)
    assert submission_table["lead_id"].is_unique
    assert submission_table["score"].between(0, 1).all()
    return submission_table

## Временная проверка улучшения

Я сравнил версии на двух последовательных пятидневных окнах. Для итоговой проверки число итераций было фиксировано заранее, без early stopping по ответам проверочного окна.

| Окно | Baseline | Версия v2 | Изменение |
|---|---:|---:|---:|
| 13–17 апреля | 0.714285 | 0.738610 | +0.024325 |
| 18–22 апреля | 0.740219 | 0.760062 | +0.019843 |

Оба окна улучшились, поэтому я использую v2 для второй отправки, сохраняя первоначальный submission отдельно.

## Дополнительные признаки и ансамбль v2

Ниже находятся все функции второй версии. Комментарии описывают, зачем я добавляю каждую группу признаков и как объединяю модели.

In [ ]:
EXTRA_WINDOWS = (0.25, 0.5, 2, 5, 10, 21, 60)


def build_extra_event_features(
    leads: pd.DataFrame,
    events_log: pd.DataFrame,
) -> pd.DataFrame:
    """Я подробнее описываю давность, короткие окна и последние переходы."""
    lead_times = leads[["lead_id", "assignment_ts"]].copy()
    lead_times["assignment_ts"] = pd.to_datetime(
        lead_times["assignment_ts"], errors="raise"
    )

    lead_history = events_log.copy()
    lead_history["event_ts"] = pd.to_datetime(
        lead_history["event_ts"], errors="raise"
    )
    lead_history = lead_history.merge(
        lead_times, on="lead_id", validate="many_to_one"
    )
    lead_history = lead_history[
        lead_history["event_ts"] < lead_history["assignment_ts"]
    ].copy()
    assert (lead_history["event_ts"] < lead_history["assignment_ts"]).all()
    lead_history["age_days"] = (
        lead_history["assignment_ts"] - lead_history["event_ts"]
    ).dt.total_seconds() / 86_400
    lead_history = lead_history.sort_values(["lead_id", "event_ts"])

    lead_ids = leads["lead_id"].astype(str)
    extra_features = pd.DataFrame(index=lead_ids)
    action_types = sorted(lead_history["event_type"].dropna().unique())

    # Для каждого типа действия я отдельно считаю давность последнего события.
    action_recency = lead_history.pivot_table(
        index="lead_id",
        columns="event_type",
        values="age_days",
        aggfunc="min",
    )
    action_recency.columns = [
        f"extra_{action}_age_min" for action in action_recency.columns
    ]
    extra_features = extra_features.join(action_recency)

    action_mean_age = lead_history.pivot_table(
        index="lead_id",
        columns="event_type",
        values="age_days",
        aggfunc="mean",
    )
    action_mean_age.columns = [
        f"extra_{action}_age_mean" for action in action_mean_age.columns
    ]
    extra_features = extra_features.join(action_mean_age)

    source_recency = lead_history.pivot_table(
        index="lead_id",
        columns="src_slot",
        values="age_days",
        aggfunc="min",
    )
    source_recency.columns = [
        f"extra_src_{str(source).replace('.', '_')}_age_min"
        for source in source_recency.columns
    ]
    extra_features = extra_features.join(source_recency)

    # Сочетание контекста и действия оказалось полезнее двух отдельных счётчиков.
    lead_history["context_action"] = (
        lead_history["ctx_seq"].astype(str)
        + "__"
        + lead_history["event_type"].astype(str)
    )
    context_action_recency = lead_history.pivot_table(
        index="lead_id",
        columns="context_action",
        values="age_days",
        aggfunc="min",
    )
    context_action_recency.columns = [
        f"extra_ctx_action_{value}_age_min"
        for value in context_action_recency.columns
    ]
    extra_features = extra_features.join(context_action_recency)

    # Я добавляю промежуточные окна, которых не было в исходной таблице.
    for window in EXTRA_WINDOWS:
        window_name = str(window).replace(".", "_")
        recent_history = lead_history[lead_history["age_days"] <= window]
        extra_features[f"extra_event_count_{window_name}d"] = (
            recent_history.groupby("lead_id").size()
        )
        action_counts = recent_history.pivot_table(
            index="lead_id",
            columns="event_type",
            values="event_ts",
            aggfunc="size",
            fill_value=0,
        ).reindex(columns=action_types, fill_value=0)
        action_counts.columns = [
            f"extra_{action}_count_{window_name}d"
            for action in action_counts.columns
        ]
        extra_features = extra_features.join(action_counts)

    # Последние переходы дают модели короткий пользовательский сценарий целиком.
    last_events = {}
    for position in (1, 2, 3):
        event_at_position = (
            lead_history.groupby("lead_id", sort=False)
            .nth(-position)
            .set_index("lead_id")
        )
        last_events[position] = event_at_position
        extra_features[f"extra_last{position}_type_ctx"] = (
            event_at_position["event_type"].astype(str)
            + "__"
            + event_at_position["ctx_seq"].astype(str)
        )
    extra_features["extra_last_type_transition"] = (
        last_events[2]["event_type"].astype(str)
        + "__"
        + last_events[1]["event_type"].astype(str)
    )
    extra_features["extra_last_ctx_transition"] = (
        last_events[2]["ctx_seq"].astype(str)
        + "__"
        + last_events[1]["ctx_seq"].astype(str)
    )

    count_columns = [
        column for column in extra_features.columns if "count" in column
    ]
    extra_features[count_columns] = extra_features[count_columns].fillna(0.0)
    return extra_features.reindex(lead_ids).reset_index(drop=True)


def add_rounded_count_features(features: pd.DataFrame) -> pd.DataFrame:
    """Я восстанавливаю целочисленную структуру трёх зашумлённых счётчиков."""
    rounded_features = {}
    noisy_columns = (
        "seller_page_views_7d",
        "seller_page_views_14d",
        "seller_page_views_30d",
    )
    for column in noisy_columns:
        rounded = features[column].round()
        rounded_features[f"{column}_rounded"] = rounded
        rounded_features[f"{column}_rounding_error"] = features[column] - rounded
        rounded_features[f"{column}_rounding_error_abs"] = (
            features[column] - rounded
        ).abs()

    for short_window, long_window in zip(noisy_columns[:-1], noisy_columns[1:]):
        rounded_features[f"{short_window}_{long_window}_rounded_increment"] = (
            features[long_window].round() - features[short_window].round()
        )
    rounded_features["seller_page_views_30d_90d_rounded_increment"] = (
        features["seller_page_views_90d"]
        - features["seller_page_views_30d"].round()
    )
    return pd.concat(
        [features.reset_index(drop=True), pd.DataFrame(rounded_features)], axis=1
    )


def prepare_enhanced_features(
    train_data: pd.DataFrame,
    test_data: pd.DataFrame,
    events_log: pd.DataFrame,
):
    """Я создаю исходный и расширенный наборы признаков одним проходом."""
    base_train, base_test, base_num_columns, base_cat_columns = prepare_features(
        train_data, test_data, events_log
    )
    all_leads = pd.concat(
        [train_data.drop(columns=[TARGET]), test_data], ignore_index=True
    )
    extra_event_features = build_extra_event_features(all_leads, events_log)
    enhanced_features = pd.concat(
        [
            pd.concat([base_train, base_test], ignore_index=True),
            extra_event_features,
        ],
        axis=1,
    )
    enhanced_features = add_rounded_count_features(enhanced_features)
    enhanced_train = enhanced_features.iloc[: len(train_data)].copy()
    enhanced_test = enhanced_features.iloc[len(train_data) :].copy()
    enhanced_cat_columns = enhanced_train.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()
    return (
        base_train,
        base_test,
        base_num_columns,
        base_cat_columns,
        enhanced_train,
        enhanced_test,
        enhanced_cat_columns,
    )


def fit_enhanced_models(
    train_features: pd.DataFrame,
    train_target: pd.Series,
    features_to_score: pd.DataFrame,
    dates_to_score: pd.Series,
    cat_columns: list[str],
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Я обучаю три устойчивые модели, выбранные на двух временных holdout."""
    train_matrix, score_matrix = make_tree_matrices(
        train_features, features_to_score, cat_columns
    )

    plain_lightgbm = LGBMClassifier(
        num_leaves=15,
        min_child_samples=60,
        learning_rate=0.02,
        n_estimators=1_800,
        reg_lambda=8.0,
        reg_alpha=0.2,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        random_state=RANDOM_STATE,
        verbosity=-1,
        n_jobs=-1,
    )
    plain_lightgbm.fit(train_matrix, train_target)
    plain_rank = rank_within_day(
        plain_lightgbm.predict_proba(score_matrix)[:, 1], dates_to_score
    )
    print("Обучена улучшенная LightGBM-модель")

    extra_trees_lightgbm = LGBMClassifier(
        num_leaves=31,
        min_child_samples=45,
        extra_trees=True,
        max_bin=127,
        learning_rate=0.02,
        n_estimators=600,
        reg_lambda=8.0,
        reg_alpha=0.2,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        random_state=RANDOM_STATE,
        verbosity=-1,
        n_jobs=-1,
    )
    extra_trees_lightgbm.fit(train_matrix, train_target)
    extra_trees_rank = rank_within_day(
        extra_trees_lightgbm.predict_proba(score_matrix)[:, 1], dates_to_score
    )
    print("Обучена Extra-Trees LightGBM-модель")

    catboost_train = train_features.reset_index(drop=True).copy()
    catboost_score_data = features_to_score.reset_index(drop=True).copy()
    for column in cat_columns:
        catboost_train[column] = (
            catboost_train[column].fillna("<пропуск>").astype(str)
        )
        catboost_score_data[column] = (
            catboost_score_data[column].fillna("<пропуск>").astype(str)
        )
    catboost_model = CatBoostClassifier(
        depth=8,
        learning_rate=0.03,
        iterations=750,
        l2_leaf_reg=8.0,
        random_strength=0.5,
        bagging_temperature=0.5,
        loss_function="Logloss",
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )
    catboost_model.fit(
        catboost_train,
        train_target,
        cat_features=cat_columns,
        verbose=False,
    )
    catboost_rank = rank_within_day(
        catboost_model.predict_proba(catboost_score_data)[:, 1], dates_to_score
    )
    print("Обучена улучшенная CatBoost-модель")
    return plain_rank, extra_trees_rank, catboost_rank


def create_submission_v2(
    train_data: pd.DataFrame,
    test_data: pd.DataFrame,
    base_train: pd.DataFrame,
    base_test: pd.DataFrame,
    base_num_columns: list[str],
    base_cat_columns: list[str],
    enhanced_train: pd.DataFrame,
    enhanced_test: pd.DataFrame,
    enhanced_cat_columns: list[str],
) -> pd.DataFrame:
    """Я объединяю старый baseline с тремя новыми моделями."""
    baseline_rank, _, _ = fit_predict_ensemble(
        train_features=base_train,
        train_target=train_data[TARGET],
        features_to_score=base_test,
        dates_to_score=test_data[DATE_COLUMN],
        num_columns=base_num_columns,
        cat_columns=base_cat_columns,
    )
    plain_rank, extra_trees_rank, catboost_rank = fit_enhanced_models(
        enhanced_train,
        train_data[TARGET],
        enhanced_test,
        test_data[DATE_COLUMN],
        enhanced_cat_columns,
    )

    # Эти простые веса устойчиво выиграли на двух непересекающихся временных окнах.
    blended_rank = (
        0.20 * baseline_rank
        + 0.30 * plain_rank
        + 0.20 * extra_trees_rank
        + 0.30 * catboost_rank
    )
    final_scores = apply_validated_rules_and_rerank(
        blended_rank, base_test, test_data[DATE_COLUMN]
    )
    submission = pd.DataFrame(
        {
            "lead_id": test_data["lead_id"].astype(str),
            "score": final_scores.astype(float),
        }
    )
    assert list(submission.columns) == ["lead_id", "score"]
    assert submission["lead_id"].is_unique
    assert submission["score"].between(0, 1).all()
    assert len(submission) == len(test_data)
    return submission

## Финальное обучение и `submission_v2.csv`

Я обучаю все компоненты на полном `train.csv`, затем ранжирую прогнозы внутри каждой даты `test.csv`. Исходный порядок `lead_id` сохраняется.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=pd.errors.PerformanceWarning)
    prepared_features = prepare_enhanced_features(train_data, test_data, events_log)

(
    base_train,
    base_test,
    base_num_columns,
    base_cat_columns,
    enhanced_train,
    enhanced_test,
    enhanced_cat_columns,
) = prepared_features

print(f"Признаков baseline: {base_train.shape[1]}")
print(f"Признаков v2: {enhanced_train.shape[1]}")

submission_v2 = create_submission_v2(
    train_data,
    test_data,
    base_train,
    base_test,
    base_num_columns,
    base_cat_columns,
    enhanced_train,
    enhanced_test,
    enhanced_cat_columns,
)

OUTPUT_PATH = Path(os.environ.get("LEAD_OUTPUT", "submission_v2.csv"))
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission_v2.to_csv(OUTPUT_PATH, index=False)

assert submission_v2["lead_id"].tolist() == test_data["lead_id"].astype(str).tolist()
assert list(submission_v2.columns) == ["lead_id", "score"]
assert submission_v2["lead_id"].is_unique
assert submission_v2["score"].between(0, 1).all()

print(f"Файл {OUTPUT_PATH} сохранён, строк: {len(submission_v2):,}")
submission_v2.head()

## Результат

`submission_v2.csv` — отдельная вторая попытка. Исходный `submission.csv`, который получил Daily AP 0.74635 на платформе, остаётся неизменным и служит безопасным baseline.